In [1]:
from pathlib import Path

import pandas as pd

from IPython.display import display


# Find the main project folder.
CURRENT_FOLDER = Path.cwd()

if CURRENT_FOLDER.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_FOLDER.parent
else:
    PROJECT_ROOT = CURRENT_FOLDER


# Folder containing CSV files from Notebooks 01–03.
PROCESSED_FOLDER = (
    PROJECT_ROOT
    / "data"
    / "processed"
)


# Change this date when processing another report.
REPORT_DATE = "2026-09-01"


# Input files required for this notebook.
INPUT_FILES = {
    "district": (
        PROCESSED_FOLDER
        / f"district_table_{REPORT_DATE}.csv"
    ),

    "district_validation": (
        PROCESSED_FOLDER
        / f"district_validation_{REPORT_DATE}.csv"
    ),

    "state": (
        PROCESSED_FOLDER
        / f"state_analysis_{REPORT_DATE}.csv"
    ),

    "locality": (
        PROCESSED_FOLDER
        / f"localities_{REPORT_DATE}.csv"
    ),

    "death": (
        PROCESSED_FOLDER
        / f"death_notes_{REPORT_DATE}.csv"
    ),
}


# Stop if any required file is missing.
for data_name, file_path in INPUT_FILES.items():

    if not file_path.exists():

        raise FileNotFoundError(
            f"Missing {data_name} file:\n"
            f"{file_path}"
        )

    print(
        f"Found {data_name}: "
        f"{file_path.name}"
    )

Found district: district_table_2026-09-01.csv
Found district_validation: district_validation_2026-09-01.csv
Found state: state_analysis_2026-09-01.csv
Found locality: localities_2026-09-01.csv
Found death: death_notes_2026-09-01.csv


In [2]:
district_wide_df = pd.read_csv(
    INPUT_FILES["district"]
)

district_validation_df = pd.read_csv(
    INPUT_FILES["district_validation"]
)

state_df = pd.read_csv(
    INPUT_FILES["state"]
)

locality_df = pd.read_csv(
    INPUT_FILES["locality"]
)

death_df = pd.read_csv(
    INPUT_FILES["death"]
)


# Compatibility fix for older Notebook 01 output.
# The district table comes from page 2.
if "source_page" not in district_wide_df.columns:

    district_wide_df.insert(
        4,
        "source_page",
        2
    )


print(
    "District rows:",
    len(district_wide_df)
)

print(
    "District validation rows:",
    len(district_validation_df)
)

print(
    "State rows:",
    len(state_df)
)

print(
    "Locality rows:",
    len(locality_df)
)

print(
    "Death-note rows:",
    len(death_df)
)

print(
    "District source page:",
    district_wide_df["source_page"].unique()
)

District rows: 14
District validation rows: 29
State rows: 26
Locality rows: 24
Death-note rows: 6
District source page: [2]


In [3]:
validation_results = []


def require(condition, check_name, details):
    """
    Record a validation result.

    If the condition is False, stop the notebook
    so that incorrect data is not trusted or saved.
    """

    passed = bool(condition)

    status = (
        "PASS"
        if passed
        else "FAIL"
    )


    validation_results.append({
        "check": check_name,
        "status": status,
        "details": details,
    })


    print(
        f"{status}: {check_name}"
    )


    if not passed:

        raise AssertionError(
            f"{check_name}\n"
            f"{details}"
        )

In [4]:
STATE_NUMERIC_COLUMNS = [
    "daily_suspected_cases",
    "daily_suspected_deaths",
    "daily_confirmed",
    "daily_deaths",

    "month_suspected_cases",
    "month_suspected_deaths",
    "month_confirmed",
    "month_deaths",

    "cumulative_suspected_cases",
    "cumulative_suspected_deaths",
    "cumulative_confirmed",
    "cumulative_deaths",
]


REQUIRED_COLUMNS = {
    "district": {
        "report_date",
        "period_type",
        "source_filename",
        "schema_version",
        "source_page",
        "district_code",
        "district_name",
    },

    "state": {
        "report_date",
        "period_type",
        "source_filename",
        "schema_version",
        "source_page",
        "disease",
        "subtype",
    } | set(STATE_NUMERIC_COLUMNS),

    "locality": {
        "report_date",
        "period_type",
        "source_filename",
        "schema_version",
        "source_page",
        "disease",
        "district_code",
        "district_name",
        "district_reported_count",
        "locality_text",
    },

    "death": {
        "report_date",
        "period_type",
        "source_filename",
        "schema_version",
        "source_page",
        "district_code",
        "district_name",
        "disease",
        "death_date",
        "parse_status",
        "raw_text",
    },
}


INPUT_TABLES = {
    "district": district_wide_df,
    "state": state_df,
    "locality": locality_df,
    "death": death_df,
}


EXPECTED_SCHEMA = {
    "district": "daily_v1",
    "state": "state_v1",
    "locality": "locality_v1",
    "death": "death_notes_v1",
}


EXPECTED_PAGE = {
    "district": 2,
    "state": 3,
    "locality": 2,
    "death": 3,
}


# Check required columns and report dates.
for table_name, dataframe in INPUT_TABLES.items():

    missing_columns = (
        REQUIRED_COLUMNS[table_name]
        - set(dataframe.columns)
    )


    require(
        not missing_columns,

        f"{table_name} required columns",

        (
            "Missing columns: "
            f"{missing_columns}"
        )
    )


    # Locality and death tables may legitimately
    # contain zero rows in some reports.
    if dataframe.empty:

        require(
            table_name in {"locality", "death"},

            f"{table_name} empty-table rule",

            (
                "Only locality and death tables "
                "may be empty."
            )
        )

        print(
            f"INFO: {table_name} table is empty."
        )

        continue


    extracted_dates = set(
        dataframe["report_date"]
        .dropna()
        .astype(str)
    )


    require(
        extracted_dates == {REPORT_DATE},

        f"{table_name} report date",

        (
            f"Expected {REPORT_DATE}; "
            f"found {extracted_dates}"
        )
    )


    period_types = set(
        dataframe["period_type"]
        .dropna()
        .astype(str)
        .str.lower()
    )


    require(
        period_types == {"daily"},

        f"{table_name} period type",

        (
            "Expected daily; "
            f"found {period_types}"
        )
    )


# Every nonempty table should come from the same PDF.
district_source_files = set(
    district_wide_df[
        "source_filename"
    ].astype(str)
)


require(
    len(district_source_files) == 1,

    "One district source PDF",

    (
        "The district data should come "
        "from one PDF."
    )
)


for table_name, dataframe in INPUT_TABLES.items():

    if dataframe.empty:
        continue


    schema_values = set(
        dataframe["schema_version"]
        .dropna()
        .astype(str)
    )


    require(
        schema_values
        == {EXPECTED_SCHEMA[table_name]},

        f"{table_name} schema version",

        (
            f"Expected "
            f"{EXPECTED_SCHEMA[table_name]}; "
            f"found {schema_values}"
        )
    )


    page_values = set(
        pd.to_numeric(
            dataframe["source_page"],
            errors="coerce"
        )
        .dropna()
        .astype(int)
    )


    require(
        page_values
        == {EXPECTED_PAGE[table_name]},

        f"{table_name} source page",

        (
            f"Expected page "
            f"{EXPECTED_PAGE[table_name]}; "
            f"found {page_values}"
        )
    )


    source_files = set(
        dataframe[
            "source_filename"
        ].astype(str)
    )


    require(
        source_files
        == district_source_files,

        f"{table_name} source PDF",

        (
            "All tables must come from "
            "the same PDF."
        )
    )

PASS: district required columns
PASS: district report date
PASS: district period type
PASS: state required columns
PASS: state report date
PASS: state period type
PASS: locality required columns
PASS: locality report date
PASS: locality period type
PASS: death required columns
PASS: death report date
PASS: death period type
PASS: One district source PDF
PASS: district schema version
PASS: district source page
PASS: district source PDF
PASS: state schema version
PASS: state source page
PASS: state source PDF
PASS: locality schema version
PASS: locality source page
PASS: locality source PDF
PASS: death schema version
PASS: death source page
PASS: death source PDF


In [5]:
COLUMN_MEANINGS = {
    "fever_op": (
        "Fever",
        "outpatient",
        None
    ),

    "fever_ip": (
        "Fever",
        "inpatient",
        None
    ),

    "chikungunya_suspected": (
        "Chikungunya",
        "suspected",
        None
    ),

    "chikungunya_confirmed": (
        "Chikungunya",
        "confirmed",
        None
    ),

    "chikungunya_deaths": (
        "Chikungunya",
        "deaths",
        None
    ),

    "dengue_suspected": (
        "Dengue",
        "suspected",
        None
    ),

    "dengue_confirmed": (
        "Dengue",
        "confirmed",
        None
    ),

    "dengue_deaths": (
        "Dengue",
        "deaths",
        None
    ),

    "leptospirosis_suspected": (
        "Leptospirosis",
        "suspected",
        None
    ),

    "leptospirosis_confirmed": (
        "Leptospirosis",
        "confirmed",
        None
    ),

    "leptospirosis_deaths": (
        "Leptospirosis",
        "deaths",
        None
    ),

    "add_confirmed": (
        "Acute Diarrhoeal Disease",
        "confirmed",
        None
    ),

    "chickenpox_confirmed": (
        "Chickenpox",
        "confirmed",
        None
    ),

    "hepatitis_a_confirmed": (
        "Hepatitis A",
        "confirmed",
        None
    ),

    "cholera_suspected_cases": (
        "Cholera",
        "suspected",
        None
    ),

    "cholera_suspected_deaths": (
        "Cholera",
        "suspected_deaths",
        None
    ),

    "cholera_confirmed_cases": (
        "Cholera",
        "confirmed",
        None
    ),

    "cholera_confirmed_deaths": (
        "Cholera",
        "deaths",
        None
    ),

    "aes_confirmed": (
        "Acute Encephalitis Syndrome",
        "confirmed",
        None
    ),

    "je_confirmed": (
        "Japanese Encephalitis",
        "confirmed",
        None
    ),

    "malaria_pv_imported": (
        "Malaria",
        "confirmed",
        "PV imported"
    ),

    "malaria_pv_indigenous": (
        "Malaria",
        "confirmed",
        "PV indigenous"
    ),

    "malaria_pf_imported": (
        "Malaria",
        "confirmed",
        "PF imported"
    ),

    "malaria_pf_indigenous": (
        "Malaria",
        "confirmed",
        "PF indigenous"
    ),

    "malaria_mixed_imported": (
        "Malaria",
        "confirmed",
        "Mixed imported"
    ),

    "malaria_mixed_indigenous": (
        "Malaria",
        "confirmed",
        "Mixed indigenous"
    ),

    "malaria_deaths": (
        "Malaria",
        "deaths",
        None
    ),

    "scrub_typhus_confirmed": (
        "Scrub Typhus",
        "confirmed",
        None
    ),

    "influenza_confirmed": (
        "Influenza",
        "confirmed",
        None
    ),
}


print(
    "Mapped district measurements:",
    len(COLUMN_MEANINGS)
)

Mapped district measurements: 29


In [10]:
EXPECTED_DISTRICT_CODES = {
    "TVM",
    "KLM",
    "PTA",
    "ALP",
    "KTM",
    "IDK",
    "EKM",
    "TSR",
    "PKD",
    "MPM",
    "KKD",
    "WYD",
    "KNR",
    "KSD",
}


DISTRICT_METADATA_COLUMNS = {
    "report_date",
    "period_type",
    "source_filename",
    "schema_version",
    "source_page",
    "district_code",
    "district_name",
}


expected_value_columns = set(
    COLUMN_MEANINGS
)


actual_value_columns = (
    set(district_wide_df.columns)
    - DISTRICT_METADATA_COLUMNS
)


missing_value_columns = (
    expected_value_columns
    - actual_value_columns
)


unexpected_value_columns = (
    actual_value_columns
    - expected_value_columns
)


require(
    len(district_wide_df) == 14,

    "Fourteen district rows",

    (
        "Kerala must have exactly "
        "14 district records."
    )
)


require(
    district_wide_df[
        "district_code"
    ].nunique() == 14,

    "Unique district codes",

    (
        "Each district code must "
        "appear exactly once."
    )
)


require(
    set(
        district_wide_df[
            "district_code"
        ]
    ) == EXPECTED_DISTRICT_CODES,

    "Expected Kerala district codes",

    (
        "The district-code set does "
        "not match Kerala's 14 districts."
    )
)


require(
    not missing_value_columns,

    "No missing district measurements",

    (
        "Missing measurements: "
        f"{missing_value_columns}"
    )
)


require(
    not unexpected_value_columns,

    "No unexpected district measurements",

    (
        "Unexpected measurements: "
        f"{unexpected_value_columns}"
    )
)


district_value_columns = list(
    COLUMN_MEANINGS
)


# Convert the district measurement columns to numbers.
district_wide_df[
    district_value_columns
] = district_wide_df[
    district_value_columns
].apply(
    pd.to_numeric,
    errors="coerce"
)


require(
    district_wide_df[
        district_value_columns
    ].notna().all().all(),

    "No missing district values",

    (
        "Every district measurement "
        "must contain a number."
    )
)


require(
    district_wide_df[
        district_value_columns
    ].ge(0).all().all(),

    "No negative district values",

    "Disease counts cannot be negative."
)


# Validate the district_validation CSV.
DISTRICT_VALIDATION_COLUMNS = {
    "column",
    "calculated_total",
    "published_total",
    "match",
}


missing_validation_columns = (
    DISTRICT_VALIDATION_COLUMNS
    - set(district_validation_df.columns)
)


require(
    not missing_validation_columns,

    "District-validation columns",

    (
        "Missing district-validation columns: "
        f"{missing_validation_columns}"
    )
)


require(
    set(
        district_validation_df["column"]
    ) == expected_value_columns,

    "All 29 totals were validated",

    (
        "The validation file must contain "
        "the same 29 district measurements."
    )
)


district_validation_df[
    "calculated_total"
] = pd.to_numeric(
    district_validation_df[
        "calculated_total"
    ],
    errors="coerce"
)


district_validation_df[
    "published_total"
] = pd.to_numeric(
    district_validation_df[
        "published_total"
    ],
    errors="coerce"
)


validation_match_values = (
    district_validation_df["match"]
    .astype(str)
    .str.lower()
    .eq("true")
)


require(
    validation_match_values.all(),

    "Published district totals match",

    (
        "At least one district sum does "
        "not match the published total."
    )
)


# Recalculate the totals again in Notebook 04.
recalculated_totals = (
    district_wide_df[
        district_value_columns
    ]
    .sum()
    .astype(int)
    .rename_axis("column")
    .reset_index(
        name="recalculated_total"
    )
)


district_totals_check_df = (
    district_validation_df.merge(
        recalculated_totals,
        on="column",
        how="outer"
    )
)


district_totals_check_df[
    "recalculation_match"
] = (
    (
        district_totals_check_df[
            "recalculated_total"
        ]
        == district_totals_check_df[
            "calculated_total"
        ]
    )
    &
    (
        district_totals_check_df[
            "recalculated_total"
        ]
        == district_totals_check_df[
            "published_total"
        ]
    )
)


require(
    district_totals_check_df[
        "recalculation_match"
    ].all(),

    "Recalculated district totals match",

    (
        "Notebook 04 recalculation did "
        "not match the published totals."
    )
)


print(
    "District total checks passed:",
    int(
        district_totals_check_df[
            "recalculation_match"
        ].sum()
    ),
    "/",
    len(district_totals_check_df)
)

PASS: Fourteen district rows
PASS: Unique district codes
PASS: Expected Kerala district codes
PASS: No missing district measurements
PASS: No unexpected district measurements
PASS: No missing district values
PASS: No negative district values
PASS: District-validation columns
PASS: All 29 totals were validated
PASS: Published district totals match
PASS: Recalculated district totals match
District total checks passed: 29 / 29


In [11]:
observation_records = []


for _, district_row in district_wide_df.iterrows():

    for source_column, meaning in COLUMN_MEANINGS.items():

        disease = meaning[0]

        metric = meaning[1]

        subtype = meaning[2]

        value = district_row[
            source_column
        ]


        observation_records.append({
            "report_date":
                str(
                    district_row[
                        "report_date"
                    ]
                ),

            "period_type":
                district_row[
                    "period_type"
                ],

            "source_filename":
                district_row[
                    "source_filename"
                ],

            "schema_version":
                district_row[
                    "schema_version"
                ],

            "source_page":
                int(
                    district_row[
                        "source_page"
                    ]
                ),

            "geography_level":
                "district",

            "district_code":
                district_row[
                    "district_code"
                ],

            "district_name":
                district_row[
                    "district_name"
                ],

            "disease":
                disease,

            "metric":
                metric,

            "subtype":
                subtype,

            "value":
                int(value),

            "source_column":
                source_column,
        })


observations_df = pd.DataFrame(
    observation_records
)


observations_df["value"] = (
    observations_df["value"]
    .astype("Int64")
)


print(
    "Long-format observations:",
    len(observations_df)
)


display(
    observations_df.head(10)
)

Long-format observations: 406


,report_date,period_type,source_filename,schema_version,source_page,geography_level,district_code,district_name,disease,metric,subtype,value,source_column
0,2026-09-01,daily,IDSP-Daily-Report-01.09.2026.pdf,daily_v1,2,district,TVM,Thiruvananthapuram,Fever,outpatient,None,786,fever_op
1,2026-09-01,daily,IDSP-Daily-Report-01.09.2026.pdf,daily_v1,2,district,TVM,Thiruvananthapuram,Fever,inpatient,None,13,fever_ip
2,2026-09-01,daily,IDSP-Daily-Report-01.09.2026.pdf,daily_v1,2,district,TVM,Thiruvananthapuram,Chikungunya,suspected,None,0,chikungunya_suspected
3,2026-09-01,daily,IDSP-Daily-Report-01.09.2026.pdf,daily_v1,2,district,TVM,Thiruvananthapuram,Chikungunya,confirmed,None,1,chikungunya_confirmed
4,2026-09-01,daily,IDSP-Daily-Report-01.09.2026.pdf,daily_v1,2,district,TVM,Thiruvananthapuram,Chikungunya,deaths,None,0,chikungunya_deaths
5,2026-09-01,daily,IDSP-Daily-Report-01.09.2026.pdf,daily_v1,2,district,TVM,Thiruvananthapuram,Dengue,suspected,None,29,dengue_suspected
6,2026-09-01,daily,IDSP-Daily-Report-01.09.2026.pdf,daily_v1,2,district,TVM,Thiruvananthapuram,Dengue,confirmed,None,20,dengue_confirmed
7,2026-09-01,daily,IDSP-Daily-Report-01.09.2026.pdf,daily_v1,2,district,TVM,Thiruvananthapuram,Dengue,deaths,None,1,dengue_deaths
8,2026-09-01,daily,IDSP-Daily-Report-01.09.2026.pdf,daily_v1,2,district,TVM,Thiruvananthapuram,Leptospirosis,suspected,None,0,leptospirosis_suspected
9,2026-09-01,daily,IDSP-Daily-Report-01.09.2026.pdf,daily_v1,2,district,TVM,Thiruvananthapuram,Leptospirosis,confirmed,None,3,leptospirosis_confirmed


In [12]:
expected_observation_count = (
    len(district_wide_df)
    * len(COLUMN_MEANINGS)
)


require(
    len(observations_df)
    == expected_observation_count,

    "Expected observation count",

    (
        f"Expected "
        f"{expected_observation_count}; "
        f"found {len(observations_df)}"
    )
)


require(
    observations_df[
        "value"
    ].notna().all(),

    "No missing observation values",

    (
        "Every normalized district "
        "observation must have a value."
    )
)


require(
    observations_df[
        "value"
    ].ge(0).all(),

    "No negative observation values",

    "Disease counts cannot be negative."
)


OBSERVATION_KEY = [
    "report_date",
    "district_code",
    "disease",
    "metric",
    "subtype",
]


duplicate_observations = (
    observations_df.duplicated(
        subset=OBSERVATION_KEY,
        keep=False
    )
)


require(
    not duplicate_observations.any(),

    "No duplicate observations",

    (
        "The same district, disease, "
        "metric and subtype must not "
        "appear more than once."
    )
)


print(
    "Validated observations:",
    len(observations_df)
)

PASS: Expected observation count
PASS: No missing observation values
PASS: No negative observation values
PASS: No duplicate observations
Validated observations: 406


In [13]:
# Convert state measurements to numeric values.
converted_state_values = (
    state_df[
        STATE_NUMERIC_COLUMNS
    ].apply(
        pd.to_numeric,
        errors="coerce"
    )
)


invalid_state_numbers = (
    state_df[
        STATE_NUMERIC_COLUMNS
    ].notna()
    &
    converted_state_values.isna()
)


require(
    not invalid_state_numbers.any().any(),

    "State numeric conversion",

    (
        "At least one nonempty state value "
        "could not be converted to a number."
    )
)


state_df[
    STATE_NUMERIC_COLUMNS
] = converted_state_values


require(
    state_df[
        "disease"
    ].fillna(
        ""
    ).str.strip().ne("").all(),

    "State disease names present",

    "Every state row needs a disease name."
)


require(
    not state_df.duplicated(
        subset=[
            "disease",
            "subtype",
        ]
    ).any(),

    "Unique state disease rows",

    (
        "A disease and subtype should "
        "appear only once."
    )
)


require(
    state_df[
        STATE_NUMERIC_COLUMNS
    ].stack().ge(0).all(),

    "No negative state values",

    "State disease counts cannot be negative."
)


# Validate locality records.
if not locality_df.empty:

    require(
        set(
            locality_df[
                "district_code"
            ].dropna()
        ).issubset(
            EXPECTED_DISTRICT_CODES
        ),

        "Valid locality district codes",

        (
            "A locality record contains "
            "an unknown district code."
        )
    )


    require(
        locality_df[
            "locality_text"
        ].fillna(
            ""
        ).str.strip().ne("").all(),

        "Locality text is present",

        (
            "Each locality row needs "
            "locality information."
        )
    )


    require(
        set(
            locality_df[
                "disease"
            ].dropna()
        ).issubset(
            set(
                state_df[
                    "disease"
                ].dropna()
            )
        ),

        "Locality diseases exist in state data",

        (
            "A locality disease is missing "
            "from the state disease table."
        )
    )


    locality_df[
        "district_reported_count"
    ] = pd.to_numeric(
        locality_df[
            "district_reported_count"
        ],
        errors="coerce"
    )


    require(
        locality_df[
            "district_reported_count"
        ].dropna().ge(0).all(),

        "No negative locality counts",

        (
            "Locality district counts "
            "cannot be negative."
        )
    )


# Validate death-note records.
if not death_df.empty:

    allowed_parse_statuses = {
        "parsed",
        "needs_review",
    }


    require(
        set(
            death_df[
                "parse_status"
            ].dropna()
        ).issubset(
            allowed_parse_statuses
        ),

        "Valid death-note statuses",

        (
            "Death-note status must be "
            "parsed or needs_review."
        )
    )


    parsed_death_df = death_df[
        death_df[
            "parse_status"
        ].eq("parsed")
    ]


    require(
        set(
            parsed_death_df[
                "district_code"
            ].dropna()
        ).issubset(
            EXPECTED_DISTRICT_CODES
        ),

        "Valid death-note district codes",

        (
            "A parsed death record contains "
            "an unknown district code."
        )
    )


    require(
        parsed_death_df[
            "disease"
        ].fillna(
            ""
        ).str.strip().ne("").all(),

        "Parsed death diseases present",

        (
            "Every parsed death record "
            "needs a disease."
        )
    )


    valid_death_dates = pd.to_datetime(
        parsed_death_df[
            "death_date"
        ],
        format="%Y-%m-%d",
        errors="coerce"
    )


    require(
        valid_death_dates.notna().all(),

        "Parsed death dates are valid",

        (
            "Every parsed death record "
            "needs a valid YYYY-MM-DD date."
        )
    )

PASS: State numeric conversion
PASS: State disease names present
PASS: Unique state disease rows
PASS: No negative state values
PASS: Valid locality district codes
PASS: Locality text is present
PASS: Locality diseases exist in state data
PASS: No negative locality counts
PASS: Valid death-note statuses
PASS: Valid death-note district codes
PASS: Parsed death diseases present
PASS: Parsed death dates are valid


In [14]:
def district_total(
    disease,
    metric,
    subtype=None
):
    """
    Add all 14 district values for one
    disease, metric and optional subtype.
    """

    selected_rows = observations_df[
        (
            observations_df[
                "disease"
            ].eq(disease)
        )
        &
        (
            observations_df[
                "metric"
            ].eq(metric)
        )
    ]


    if subtype is None:

        selected_rows = selected_rows[
            selected_rows[
                "subtype"
            ].isna()
        ]

    else:

        selected_rows = selected_rows[
            selected_rows[
                "subtype"
            ].eq(subtype)
        ]


    return int(
        selected_rows[
            "value"
        ].sum()
    )


def state_value(
    disease,
    value_column,
    subtype=None
):
    """
    Get one value from the statewide
    disease-analysis table.
    """

    selected_rows = state_df[
        state_df[
            "disease"
        ].eq(disease)
    ]


    if subtype is None:

        selected_rows = selected_rows[
            selected_rows[
                "subtype"
            ].isna()
        ]

    else:

        selected_rows = selected_rows[
            selected_rows[
                "subtype"
            ].eq(subtype)
        ]


    if len(selected_rows) != 1:

        raise ValueError(
            "Expected exactly one state row for:\n"
            f"Disease: {disease}\n"
            f"Subtype: {subtype}\n"
            f"Rows found: {len(selected_rows)}"
        )


    value = selected_rows.iloc[0][
        value_column
    ]


    if pd.isna(value):

        raise ValueError(
            "Missing state value for:\n"
            f"Disease: {disease}\n"
            f"Subtype: {subtype}\n"
            f"Column: {value_column}"
        )


    return int(value)


# Combine PV, PF and Mixed imported malaria.
malaria_imported_total = int(
    observations_df[
        (
            observations_df[
                "disease"
            ].eq("Malaria")
        )
        &
        (
            observations_df[
                "metric"
            ].eq("confirmed")
        )
        &
        (
            observations_df[
                "subtype"
            ]
            .fillna("")
            .str.endswith("imported")
        )
    ][
        "value"
    ].sum()
)


# Combine PV, PF and Mixed indigenous malaria.
malaria_indigenous_total = int(
    observations_df[
        (
            observations_df[
                "disease"
            ].eq("Malaria")
        )
        &
        (
            observations_df[
                "metric"
            ].eq("confirmed")
        )
        &
        (
            observations_df[
                "subtype"
            ]
            .fillna("")
            .str.endswith("indigenous")
        )
    ][
        "value"
    ].sum()
)


CROSS_PAGE_CHECKS = [
    (
        "Fever OP",
        district_total(
            "Fever",
            "outpatient"
        ),
        state_value(
            "Fever",
            "daily_confirmed"
        ),
    ),

    (
        "Chikungunya confirmed",
        district_total(
            "Chikungunya",
            "confirmed"
        ),
        state_value(
            "Chikungunya",
            "daily_confirmed"
        ),
    ),

    (
        "Dengue suspected",
        district_total(
            "Dengue",
            "suspected"
        ),
        state_value(
            "Dengue",
            "daily_suspected_cases"
        ),
    ),

    (
        "Dengue confirmed",
        district_total(
            "Dengue",
            "confirmed"
        ),
        state_value(
            "Dengue",
            "daily_confirmed"
        ),
    ),

    (
        "Dengue deaths",
        district_total(
            "Dengue",
            "deaths"
        ),
        state_value(
            "Dengue",
            "daily_deaths"
        ),
    ),

    (
        "Leptospirosis suspected",
        district_total(
            "Leptospirosis",
            "suspected"
        ),
        state_value(
            "Leptospirosis",
            "daily_suspected_cases"
        ),
    ),

    (
        "Leptospirosis confirmed",
        district_total(
            "Leptospirosis",
            "confirmed"
        ),
        state_value(
            "Leptospirosis",
            "daily_confirmed"
        ),
    ),

    (
        "Leptospirosis deaths",
        district_total(
            "Leptospirosis",
            "deaths"
        ),
        state_value(
            "Leptospirosis",
            "daily_deaths"
        ),
    ),

    (
        "ADD confirmed",
        district_total(
            "Acute Diarrhoeal Disease",
            "confirmed"
        ),
        state_value(
            "Acute Diarrhoeal Disease",
            "daily_confirmed"
        ),
    ),

    (
        "Chickenpox confirmed",
        district_total(
            "Chickenpox",
            "confirmed"
        ),
        state_value(
            "Chickenpox",
            "daily_confirmed"
        ),
    ),

    (
        "Hepatitis A confirmed",
        district_total(
            "Hepatitis A",
            "confirmed"
        ),
        state_value(
            "Hepatitis A",
            "daily_confirmed"
        ),
    ),

    (
        "Scrub Typhus confirmed",
        district_total(
            "Scrub Typhus",
            "confirmed"
        ),
        state_value(
            "Scrub Typhus",
            "daily_confirmed"
        ),
    ),

    (
        "Influenza confirmed",
        district_total(
            "Influenza",
            "confirmed"
        ),
        state_value(
            "Influenza",
            "daily_confirmed"
        ),
    ),

    (
        "Malaria imported",
        malaria_imported_total,
        state_value(
            "Malaria",
            "daily_confirmed",
            "imported"
        ),
    ),

    (
        "Malaria indigenous",
        malaria_indigenous_total,
        state_value(
            "Malaria",
            "daily_confirmed",
            "indigenous"
        ),
    ),
]


cross_validation_df = pd.DataFrame(
    CROSS_PAGE_CHECKS,
    columns=[
        "check",
        "district_total",
        "state_total",
    ]
)


cross_validation_df["match"] = (
    cross_validation_df[
        "district_total"
    ]
    ==
    cross_validation_df[
        "state_total"
    ]
)


require(
    cross_validation_df[
        "match"
    ].all(),

    "All district-versus-state checks",

    (
        "At least one district total does "
        "not match the statewide value."
    )
)


print(
    "Cross-page checks passed:",
    int(
        cross_validation_df[
            "match"
        ].sum()
    ),
    "/",
    len(cross_validation_df)
)


display(
    cross_validation_df
)

PASS: All district-versus-state checks
Cross-page checks passed: 15 / 15


,check,district_total,state_total,match
0,Fever OP,12480,12480,True
1,Chikungunya confirmed,1,1,True
2,Dengue suspected,174,174,True
3,Dengue confirmed,79,79,True
4,Dengue deaths,1,1,True
5,Leptospirosis suspected,11,11,True
6,Leptospirosis confirmed,18,18,True
7,Leptospirosis deaths,2,2,True
8,ADD confirmed,1793,1793,True
9,Chickenpox confirmed,83,83,True


In [15]:
LOCALITY_VALIDATION_COLUMNS = [
    "disease",
    "district_code",
    "locality_count",
    "district_confirmed",
    "checked",
    "match",
]


locality_validation_records = []


explicit_locality_rows = locality_df[
    locality_df[
        "district_reported_count"
    ].notna()
]


for _, locality_row in explicit_locality_rows.iterrows():

    matching_observations = observations_df[
        (
            observations_df[
                "district_code"
            ].eq(
                locality_row[
                    "district_code"
                ]
            )
        )
        &
        (
            observations_df[
                "disease"
            ].eq(
                locality_row[
                    "disease"
                ]
            )
        )
        &
        (
            observations_df[
                "metric"
            ].eq("confirmed")
        )
        &
        (
            observations_df[
                "subtype"
            ].isna()
        )
    ]


    locality_count = int(
        locality_row[
            "district_reported_count"
        ]
    )


    # Check only when the district table has
    # exactly one matching confirmed value.
    if len(matching_observations) == 1:

        district_confirmed = int(
            matching_observations.iloc[0][
                "value"
            ]
        )

        checked = True

        match = (
            locality_count
            == district_confirmed
        )

    else:

        district_confirmed = pd.NA

        checked = False

        match = pd.NA


    locality_validation_records.append({
        "disease":
            locality_row["disease"],

        "district_code":
            locality_row["district_code"],

        "locality_count":
            locality_count,

        "district_confirmed":
            district_confirmed,

        "checked":
            checked,

        "match":
            match,
    })


locality_count_validation_df = pd.DataFrame(
    locality_validation_records,
    columns=LOCALITY_VALIDATION_COLUMNS
)


checked_locality_rows = (
    locality_count_validation_df[
        locality_count_validation_df[
            "checked"
        ].eq(True)
    ]
)


if checked_locality_rows.empty:

    print(
        "INFO: No explicit locality counts "
        "were available for comparison."
    )

else:

    require(
        checked_locality_rows[
            "match"
        ].eq(True).all(),

        "Explicit locality counts match",

        (
            "At least one explicit locality "
            "count does not match its "
            "district confirmed value."
        )
    )


print(
    "Locality counts checked:",
    len(checked_locality_rows)
)


display(
    locality_count_validation_df
)

PASS: Explicit locality counts match
Locality counts checked: 10


,disease,district_code,locality_count,district_confirmed,checked,match
0,Dengue,TVM,20,20,True,True
1,Dengue,KLM,11,11,True,True
2,Dengue,PTA,3,3,True,True
3,Dengue,IDK,1,1,True,True
4,Dengue,EKM,7,7,True,True
5,Dengue,TSR,22,22,True,True
6,Dengue,PKD,3,3,True,True
7,Dengue,MPM,10,10,True,True
8,Dengue,WYD,1,1,True,True
9,Dengue,KNR,1,1,True,True


In [16]:
if death_df.empty:

    death_review_df = death_df.copy()

else:

    death_review_df = death_df[
        death_df[
            "parse_status"
        ].eq("needs_review")
    ].copy()


if death_review_df.empty:

    print(
        "Death notes requiring review: 0"
    )

else:

    print(
        "WARNING: Death notes requiring review:",
        len(death_review_df)
    )


validation_summary_df = pd.DataFrame(
    validation_results
)


all_checks_passed = (
    validation_summary_df[
        "status"
    ].eq("PASS").all()
)


print(
    "All validation checks passed:",
    all_checks_passed
)

print(
    "Total recorded checks:",
    len(validation_summary_df)
)


display(
    validation_summary_df
)

Death notes requiring review: 0
All validation checks passed: True
Total recorded checks: 67


,check,status,details
0,district required columns,PASS,Missing columns: set()
1,district report date,PASS,Expected 2026-09-01; found {'2026-09-01'}
2,district period type,PASS,Expected daily; found {'daily'}
3,state required columns,PASS,Missing columns: set()
4,state report date,PASS,Expected 2026-09-01; found {'2026-09-01'}
...,...,...,...
62,Valid death-note district codes,PASS,A parsed death record contains an unknown dist...
63,Parsed death diseases present,PASS,Every parsed death record needs a disease.
64,Parsed death dates are valid,PASS,Every parsed death record needs a valid YYYY-M...
65,All district-versus-state checks,PASS,At least one district total does not match the...


In [17]:
if not all_checks_passed:

    raise RuntimeError(
        "Trusted outputs were not saved "
        "because validation failed."
    )


OUTPUT_FILES = {
    "district_observations": (
        PROCESSED_FOLDER
        / (
            "trusted_district_observations_"
            f"{REPORT_DATE}.csv"
        )
    ),

    "state_analysis": (
        PROCESSED_FOLDER
        / (
            "trusted_state_analysis_"
            f"{REPORT_DATE}.csv"
        )
    ),

    "localities": (
        PROCESSED_FOLDER
        / (
            "trusted_localities_"
            f"{REPORT_DATE}.csv"
        )
    ),

    "death_notes": (
        PROCESSED_FOLDER
        / (
            "trusted_death_notes_"
            f"{REPORT_DATE}.csv"
        )
    ),

    "cross_validation": (
        PROCESSED_FOLDER
        / (
            "cross_validation_"
            f"{REPORT_DATE}.csv"
        )
    ),

    "locality_validation": (
        PROCESSED_FOLDER
        / (
            "locality_count_validation_"
            f"{REPORT_DATE}.csv"
        )
    ),

    "validation_summary": (
        PROCESSED_FOLDER
        / (
            "validation_summary_"
            f"{REPORT_DATE}.csv"
        )
    ),
}


observations_df.to_csv(
    OUTPUT_FILES[
        "district_observations"
    ],
    index=False
)


state_df.to_csv(
    OUTPUT_FILES[
        "state_analysis"
    ],
    index=False
)


locality_df.to_csv(
    OUTPUT_FILES[
        "localities"
    ],
    index=False
)


death_df.to_csv(
    OUTPUT_FILES[
        "death_notes"
    ],
    index=False
)


cross_validation_df.to_csv(
    OUTPUT_FILES[
        "cross_validation"
    ],
    index=False
)


locality_count_validation_df.to_csv(
    OUTPUT_FILES[
        "locality_validation"
    ],
    index=False
)


validation_summary_df.to_csv(
    OUTPUT_FILES[
        "validation_summary"
    ],
    index=False
)


# Save a separate review file only if
# an unparsed death note exists.
if not death_review_df.empty:

    death_review_file = (
        PROCESSED_FOLDER
        / (
            "death_notes_review_"
            f"{REPORT_DATE}.csv"
        )
    )

    death_review_df.to_csv(
        death_review_file,
        index=False
    )

    print(
        "Saved review file:",
        death_review_file.name
    )


print("\nTrusted files saved:\n")


for output_name, output_path in OUTPUT_FILES.items():

    print(
        f"{output_name}: "
        f"{output_path.name}"
    )


Trusted files saved:

district_observations: trusted_district_observations_2026-09-01.csv
state_analysis: trusted_state_analysis_2026-09-01.csv
localities: trusted_localities_2026-09-01.csv
death_notes: trusted_death_notes_2026-09-01.csv
cross_validation: cross_validation_2026-09-01.csv
locality_validation: locality_count_validation_2026-09-01.csv
validation_summary: validation_summary_2026-09-01.csv
